In [4]:
import os
import time
from datetime import datetime
from IPython.display import clear_output

# 1. 完整任务字典
TASKS = {
    "3.4.1": 6000, "3.4.2": 6000, "3.4.3": 4000, "3.4.4": 4000, "3.4.5": 4000, "3.4.6": 3000, "3.4.7": 3000,
    
    "5.4.1": 6000, "5.4.2": 6000, "5.4.3": 5000, "5.4.4": 6000, "5.4.5": 5000, "5.4.6": 4000, "5.4.7": 4000, "5.4.8": 5000, "5.4.9": 5000, "5.4.10": 4000,
    
    "5.7.1": 7500, "5.7.2": 6000, "5.7.3": 5000, "5.7.4": 4000, "5.7.5": 4000, "5.7.6": 5000, "5.7.7": 5000, "5.7.8": 6000, "5.7.9": 5000, "5.7.10": 5000, "5.7.11": 4000, "5.7.12": 3500,
    
    "6.1.1": 10000, "6.1.2": 9000, "6.1.3": 7500, "6.1.4": 7500, "6.1.5": 7500, "6.1.6": 9000, "6.1.7": 7500, "6.1.8": 7000, 
    
    "6.2.1": 9000, "6.2.2": 7500, "6.2.3": 6000, "6.2.4": 7500, "6.2.5": 6000, "6.2.6": 6000, "6.2.7": 6000, "6.2.8": 7000,
    
    "6.3.1": 9000, "6.3.2": 9000, "6.3.3": 7500, "6.3.4": 7500, "6.3.5": 6000, "6.3.6": 6000, "6.3.7": 9000, "6.3.8": 6000,
    
    "6.4.1": 6000, "6.4.2": 6000, "6.4.3": 6000, "6.4.4": 5000, "6.4.5": 6000, "6.4.6": 5000, "6.4.7": 5000, "6.4.8": 6000
}

TARGET_TOTAL = sum(TASKS.values())
DATA_DIR = "output/data"

CATEGORY_NAMES = {
    "3.4": "✍️ 3.4 转写任务",
    "5.4": "⚖️ 5.4 法律行政",
    "5.7": "🎭 5.7 文化艺术",
    "6.1": "🛡️ 6.1 拒绝与边界",
    "6.2": "✅ 6.2 诚实与准确",
    "6.3": "🤝 6.3 价值观与尊重",
    "6.4": "💬 6.4 对话质量"
}

def make_bar(current, total, length=15):
    """生成字符串进度条"""
    if total == 0: return '-' * length
    safe_current = min(current, total)
    filled = int(length * safe_current // total)
    return '█' * filled + '-' * (length - filled)

def get_progress_data():
    base_dir = DATA_DIR if os.path.exists(DATA_DIR) else f"../{DATA_DIR}"
    task_counts = {}
    total_count = 0
    
    for task in TASKS.keys():
        file_path = os.path.join(base_dir, task, "data.jsonl")
        count = 0
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                count = sum(1 for _ in f)
        task_counts[task] = count
        total_count += count
        
    return total_count, task_counts

# --- 核心逻辑 ---

current_count, task_counts = get_progress_data()
current_time = time.time()

if 'last_check_time' not in globals():
    last_check_time = current_time
    last_check_count = current_count
    is_first_run = True
else:
    is_first_run = False

time_diff = current_time - last_check_time
count_diff = current_count - last_check_count
speed_per_hour = 0
if time_diff > 0.5 and not is_first_run:
    speed_per_hour = count_diff / (time_diff / 3600)

# --- 渲染大盘 ---

clear_output(wait=True)
print("="*90)
print(f"🚀 藏文 SFT 数据合成大盘 | 刷新时间: {datetime.now().strftime('%H:%M:%S')}")
print("="*90)

global_pct = (current_count / TARGET_TOTAL) * 100
global_bar = make_bar(current_count, TARGET_TOTAL, length=40)
print(f"🎯 总体进度: [{global_bar}] {global_pct:.2f}%")
print(f"✅ 当前汇总: {current_count:,} / {TARGET_TOTAL:,} 条")

if not is_first_run:
    print(f"📈 增量动态: \033[1;32m+{count_diff:,}\033[0m 条 | ⚡ 实时时速: \033[1;36m{speed_per_hour:,.0f}\033[0m 条/小时")

print("="*90)

# 将任务按前缀(如 "6.1")分组并计算合计
grouped_tasks = {}
category_stats = {}

for t, target in TASKS.items():
    prefix = ".".join(t.split(".")[:2])
    if prefix not in grouped_tasks:
        grouped_tasks[prefix] = []
        category_stats[prefix] = {"current": 0, "target": 0}
    
    grouped_tasks[prefix].append(t)
    category_stats[prefix]["current"] += task_counts[t]
    category_stats[prefix]["target"] += target

# 渲染分组详情
for prefix, name in CATEGORY_NAMES.items():
    stats = category_stats.get(prefix)
    if not stats: continue
    
    # 打印分类标题及分类合计统计
    cat_pct = (stats["current"] / stats["target"]) * 100
    cat_bar = make_bar(stats["current"], stats["target"], length=15)
    print(f"\n{name:<25} 合计: {stats['current']:>6}/{stats['target']:<6} | {cat_pct:>5.1f}% [{cat_bar}]")
    print("-" * 90)
    
    tasks_in_group = grouped_tasks[prefix]
    # 两两一组打印
    for i in range(0, len(tasks_in_group), 2):
        t1 = tasks_in_group[i]
        c1, t_t1 = task_counts[t1], TASKS[t1]
        p1 = (c1 / t_t1) * 100
        # 针对完成的显示绿色
        color1 = "\033[1;32m" if p1 >= 100 else ""
        col1 = f"{color1}{t1:<6} [{make_bar(c1, t_t1, 8)}] {p1:>5.1f}% ({c1:>5}/{t_t1})\033[0m"
        
        if i + 1 < len(tasks_in_group):
            t2 = tasks_in_group[i+1]
            c2, t_t2 = task_counts[t2], TASKS[t2]
            p2 = (c2 / t_t2) * 100
            color2 = "\033[1;32m" if p2 >= 100 else ""
            col2 = f"{color2}{t2:<6} [{make_bar(c2, t_t2, 8)}] {p2:>5.1f}% ({c2:>5}/{t_t2})\033[0m"
            print(f"{col1:<52} |  {col2}")
        else:
            print(col1)

# 更新状态记录
last_check_time = current_time
last_check_count = current_count
print("\n" + "="*90)
print("💡 提示: 选中单元格按 [Shift + Enter] 刷新。已完成 100% 的子任务将显示为绿色。")

🚀 藏文 SFT 数据合成大盘 | 刷新时间: 18:03:39
🎯 总体进度: [██████████████████████████████----------] 77.28%
✅ 当前汇总: 282,071 / 365,000 条
📈 增量动态: +1,934 条 | ⚡ 实时时速: 5,461 条/小时

✍️ 3.4 转写任务               合计:   3687/30000  |  12.3% [█--------------]
------------------------------------------------------------------------------------------
3.4.1  [--------]   2.6% (  159/6000)            |  3.4.2  [--------]   5.3% (  316/6000)
3.4.3  [--------]   9.3% (  371/4000)            |  3.4.4  [--------]   6.3% (  252/4000)
3.4.5  [███-----]  48.7% ( 1948/4000)            |  3.4.6  [█-------]  19.7% (  590/3000)
3.4.7  [--------]   1.7% (   51/3000)

⚖️ 5.4 法律行政               合计:  41689/50000  |  83.4% [████████████---]
------------------------------------------------------------------------------------------
5.4.1  [████----]  61.6% ( 3693/6000)            |  5.4.2  [████----]  58.7% ( 3520/6000)
5.4.3  [███████-]  92.2% ( 4612/5000)            |  5.4.4  [████----]  62.5% ( 3749/6000)
5.4.5  [███████-]  92.8% ( 46